# 03 — Earth observation GeoAI: Sentinel-2 + Copernicus DEM + WorldCover

**[🚀 Launch this notebook live](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/lab/index.html?path=03_earth_observation_geoai.ipynb)**

This lab uses one aligned chip from **Major TOM Core**, an AI-ready multimodal dataset hosted on Source Cooperative. The same tile index exposes Sentinel-2, Copernicus DEM, Landsat, and ESA WorldCover.

We first stream layers into GeoLibre, then download only the Sentinel-2 chip into the browser and run a compact unsupervised raster classification with scikit-learn.

In [ ]:
import sys
if sys.platform == "emscripten":
    import micropip
    await micropip.install("geolibre==3.0.0")
from geolibre import Map

In [ ]:
BASE = "https://data.source.coop/major-tom/core/DATA/42"
S2 = f"{BASE}/s2/data.tif"
DEM = f"{BASE}/dem/data.tif"
WC = f"{BASE}/wc/data.tif"

m = Map(center=(8.0, 48.0), zoom=7, height="720px")
m.add_cog(S2, name="Sentinel-2 RGB", bands=[4, 3, 2], rescale=[0, 3000])
m.add_cog(DEM, name="Copernicus DEM", colormap="terrain")
m.add_cog(WC, name="ESA WorldCover")
m

## Browser-native unsupervised land-cover segmentation

The following cell reads the COG into memory through ordinary browser HTTP, downsamples it for responsiveness, uses red/green/blue/NIR reflectance as features, and runs K-Means. This is intentionally small enough for a JupyterLite teaching example.

In [ ]:
import sys, io, requests, numpy as np
if sys.platform == "emscripten":
    import pyodide_http
    pyodide_http.patch_all()

import rasterio
from rasterio.io import MemoryFile
from rasterio.enums import Resampling
from rasterio.features import shapes
from affine import Affine
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import MiniBatchKMeans
import geopandas as gpd
from shapely.geometry import shape

blob = requests.get(S2, timeout=120).content

with MemoryFile(blob) as mem:
    with mem.open() as src:
        # Downsample to keep browser computation snappy.
        h = min(160, src.height)
        w = min(160, src.width)
        arr = src.read(
            [4, 3, 2, 8],
            out_shape=(4, h, w),
            resampling=Resampling.bilinear,
        ).astype("float32")
        scale_x = src.width / w
        scale_y = src.height / h
        transform = src.transform * Affine.scale(scale_x, scale_y)
        crs = src.crs

pixels = np.moveaxis(arr, 0, -1).reshape(-1, 4)
valid = np.isfinite(pixels).all(axis=1) & (pixels.sum(axis=1) > 0)
X = pixels[valid]
Xz = StandardScaler().fit_transform(X)

model = MiniBatchKMeans(n_clusters=6, random_state=42, n_init=10, batch_size=2048)
labels_valid = model.fit_predict(Xz)

labels = np.full(len(pixels), -1, dtype="int16")
labels[valid] = labels_valid
labels = labels.reshape(h, w)

labels.shape, np.unique(labels, return_counts=True)

In [ ]:
# Polygonize classified pixels and dissolve each class.
records = []
mask = labels >= 0
for geom, value in shapes(labels, mask=mask, transform=transform):
    records.append({"cluster": int(value), "geometry": shape(geom)})

classes = gpd.GeoDataFrame(records, crs=crs).dissolve(by="cluster").reset_index()
classes = classes.to_crs(4326)
classes

In [ ]:
m.add_choropleth(
    classes,
    column="cluster",
    name="Browser K-Means EO classes",
    class_count=6,
    colormap="turbo",
    scheme="equal-interval",
    fillOpacity=0.45,
)
m

## Interpretation

These clusters are spectral groupings, not semantic land-cover labels. A stronger workflow would compare them with ESA WorldCover, construct training samples, assess accuracy, and then move to supervised or deep-learning models. The full GeoAI package is appropriate when you need PyTorch models, segmentation architectures, GPU inference, or large training pipelines.

## Data & software citations

- Major TOM Core: https://source.coop/major-tom/core
- Sentinel-2: Copernicus / European Union; accessed through the Major TOM Source Cooperative mirror.
- Copernicus DEM GLO-30: accessed through Major TOM; follow the source collection's Copernicus DEM attribution/license.
- ESA WorldCover: accessed through Major TOM; follow ESA WorldCover citation/license guidance.
- Source Cooperative: https://source.coop/
- GeoAI: Wu, Q. (2026), JOSS 11(118), 9605. https://doi.org/10.21105/joss.09605
- GeoLibre: https://geolibre.app/